In [ ]:
# Code for FID Score calculation using RADIMAGENET and Classifier

# ── Safe remount — always run this first ─────────────────────────────
import os, shutil, subprocess
from google.colab import drive

subprocess.run(['fusermount', '-uz', '/content/drive'], capture_output=True)
shutil.rmtree('/content/drive', ignore_errors=True)
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
 # ──── Install and Imports ────────────────────────────────────

!pip install torch torchvision scikit-learn -q

import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from collections import Counter
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [ ]:
# ── Paths ────
HEALTHY_DIR    = '/content/drive/MyDrive/Dissertation2026/oasis-coronal-healthy'
ALZ_REAL_DIR   = '/content/drive/MyDrive/Dissertation2026/oasis-coronal-alzheimers'
ALZ_SYNTH_DIR = '/content/drive/MyDrive/Dissertation2026/ddpm-coronal-v7_output/fid_generated_350_ddpm'  # 350 DDPM synthetic images

# ───── CLASSIFIER TRAINING configs───────────────────────────

IMAGE_SIZE    = 128
BATCH_SIZE    = 16
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-4
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Quick verification
print(f"Healthy images   : {len(os.listdir(HEALTHY_DIR))}")
print(f"Real Alzheimer's : {len(os.listdir(ALZ_REAL_DIR))}")
print(f"Synthetic        : {len(os.listdir(ALZ_SYNTH_DIR))}")

Healthy images   : 2352
Real Alzheimer's : 700
Synthetic        : 350


In [ ]:
def get_subject_id(filename):
    match = re.match(r'(OAS1_\d+_MR\d+)', filename)
    return match.group(1) if match else None

# Quick test — should print a subject ID
print(get_subject_id(os.listdir(HEALTHY_DIR)[0]))

OAS1_0258_MR1


In [ ]:
 # ──── TRAINING AND TEST SPLIT ────────────────────────────────────────

def split_by_subject(directory, test_size=0.2, seed=42):                                # code refined by Claude AI
    files = [f for f in os.listdir(directory) if f.endswith('.png')]
    subject_files = {}
    for f in files:
        sid = get_subject_id(f)
        subject_files.setdefault(sid, []).append(f)
    subjects = list(subject_files.keys())
    train_subs, test_subs = train_test_split(
        subjects, test_size=test_size, random_state=seed
    )
    train_files = [f for s in train_subs for f in subject_files[s]]
    test_files  = [f for s in test_subs  for f in subject_files[s]]
    return train_files, test_files

healthy_train, healthy_test = split_by_subject(HEALTHY_DIR)
alz_train,     alz_test     = split_by_subject(ALZ_REAL_DIR)
synth_files = [f for f in os.listdir(ALZ_SYNTH_DIR) if f.endswith('.png')]

print(f"Healthy   — train: {len(healthy_train)}, test: {len(healthy_test)}")
print(f"Alzheimer — train: {len(alz_train)}, test: {len(alz_test)}")
print(f"Synthetic — train only: {len(synth_files)}")

Healthy   — train: 1876, test: 476
Alzheimer — train: 560, test: 140
Synthetic — train only: 350


In [ ]:
 # ──── TEST SET config ────────────────────────────────────────

transform = transforms.Compose([                                                        # code refined by Claude AI
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

class MRIClassifierDataset(Dataset):
    def __init__(self, healthy_files, healthy_dir,
                 alz_files, alz_dir, transform=None):
        self.samples = []
        for f in healthy_files:
            self.samples.append((os.path.join(healthy_dir, f), 0))
        for f in alz_files:
            self.samples.append((os.path.join(alz_dir, f), 1))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

class CombinedDataset(Dataset):
    def __init__(self, healthy_files, healthy_dir,
                 alz_real_files, alz_real_dir,
                 alz_synth_files, alz_synth_dir, transform=None):
        self.samples = []
        for f in healthy_files:
            self.samples.append((os.path.join(healthy_dir, f), 0))
        for f in alz_real_files:
            self.samples.append((os.path.join(alz_real_dir, f), 1))
        for f in alz_synth_files:
            self.samples.append((os.path.join(alz_synth_dir, f), 1))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

# Fixed test set — real data only, same for all experiments
test_dataset = MRIClassifierDataset(
    healthy_test, HEALTHY_DIR,
    alz_test, ALZ_REAL_DIR,
    transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Test set: {len(test_dataset)} images")

Test set: 616 images


In [ ]:
# ──── Classifier Variants ────────────────────────────────────────

dataset_A = MRIClassifierDataset(
    healthy_train, HEALTHY_DIR,
    alz_train, ALZ_REAL_DIR,
    transform=transform
)
dataset_B = CombinedDataset(
    healthy_train, HEALTHY_DIR,
    alz_train, ALZ_REAL_DIR,
    synth_files, ALZ_SYNTH_DIR,
    transform=transform
)
dataset_C = MRIClassifierDataset(
    healthy_train, HEALTHY_DIR,
    synth_files, ALZ_SYNTH_DIR,
    transform=transform
)

print(f"Config A (Real only)       : {len(dataset_A)}")
print(f"Config B (Real+Synthetic)  : {len(dataset_B)}")
print(f"Config C (Synthetic only)  : {len(dataset_C)}")

Config A (Real only)       : 2436
Config B (Real+Synthetic)  : 2786
Config C (Synthetic only)  : 2226


In [1]:
# ──── Radimagenet Classifier importing to local ResNet50 ────────────────────────────────────────
                                            # code refined by Claude AI: Radimagenet weights did not load properly
                                            # So I had to load the weights into a local ResNet50 model
!pip install huggingface_hub -q

from torchvision import models
from huggingface_hub import hf_hub_download
import torch
import torch.nn as nn


class RadImageNetClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        # Create empty ResNet50 architecture
        self.model = models.resnet50(weights=None)
        # Download RadImageNet weights
        model_path = hf_hub_download(
            repo_id="Lab-Rasool/RadImageNet",
            filename="ResNet50.pt"
        )
        state_dict = torch.load(
            model_path,
            map_location="cpu"
        )

       # Remap Sequential indices to torchvision's named layers
        index_to_name = {'0': 'conv1', '1': 'bn1', '4': 'layer1',
                          '5': 'layer2', '6': 'layer3', '7': 'layer4'}
        new_state_dict = {}
        for key, value in state_dict.items():
            if not key.startswith("backbone."):
                continue
            parts = key.replace("backbone.", "").split(".")
            if parts[0] in index_to_name:
                parts[0] = index_to_name[parts[0]]
                new_state_dict[".".join(parts)] = value

        # Load pretrained medical weights
        missing, unexpected = self.model.load_state_dict(
            new_state_dict,
            strict=False
        )
        print("RadImageNet loaded")
        print("Missing:", missing[:5])
        print("Unexpected:", unexpected[:5])

        # Adapt RGB -> grayscale
        original_conv = self.model.conv1

        self.model.conv1 = nn.Conv2d(
            1,
            64,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )
        # Copy RGB weights averaged into grayscale
        with torch.no_grad():
            self.model.conv1.weight = nn.Parameter(
                original_conv.weight.mean(
                    dim=1,
                    keepdim=True
                )
            )
        # Binary classifier
        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            2
        )
    def forward(self, x):
        return self.model(x)

print("RadImageNet classifier ready ✅")

RadImageNet classifier ready ✅


In [ ]:
# ──── Training functions ────────────────────────────────────────

def get_class_weights(dataset):
    labels = [label for _, label in dataset.samples]
    counts = Counter(labels)
    total  = len(labels)
    w0 = total / (2 * counts[0])
    w1 = total / (2 * counts[1])
    return torch.tensor([w0, w1], dtype=torch.float32).to(device)

def train_model(train_dataset, model, name, lr=LEARNING_RATE):
    print(f"\n{'='*50}\nTraining: {name}\n{'='*50}")
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    # optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # AdamW better optimizer for fine-tuning pretrained vision models.
    optimizer = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    class_weights = get_class_weights(train_dataset)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f"Class weights — healthy: {class_weights[0]:.3f}, alzheimer: {class_weights[1]:.3f}")

    for epoch in range(NUM_EPOCHS):          # function refined by Claude AI
        model.train()
        epoch_loss = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch+1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {epoch_loss/len(loader):.4f}")
    return model

def evaluate_model(model, test_loader, name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            preds = torch.argmax(model(images.to(device)), dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    cm   = confusion_matrix(all_labels, all_preds)

    print(f"\n--- {name} ---")
    print(f"Accuracy : {acc:.3f}  Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}  F1       : {f1:.3f}")
    print(f"Confusion Matrix:\n{cm}")
    return {'name': name, 'accuracy': acc,
            'precision': prec, 'recall': rec, 'f1': f1}


In [ ]:
results = []

# ── RadImageNet experiments ───────────────────────────────

rad_a = train_model(
    dataset_A,
    RadImageNetClassifier().to(device),
    "RadImageNet Config A: Real Only",
    lr=1e-5
)
results.append(
    evaluate_model(
        rad_a,
        test_loader,
        "RadImageNet Config A"
    )
)
rad_b = train_model(
    dataset_B,
    RadImageNetClassifier().to(device),
    "RadImageNet Config B: Real + Synthetic",
    lr=1e-5
)
results.append(
    evaluate_model(
        rad_b,
        test_loader,
        "RadImageNet Config B"
    )
)
rad_c = train_model(
    dataset_C,
    RadImageNetClassifier().to(device),
    "RadImageNet Config C: Synthetic Only",
    lr=1e-5
)
results.append(
    evaluate_model(
        rad_c,
        test_loader,
        "RadImageNet Config C"
    )
)

summary_rad = pd.DataFrame(results)
print(summary_rad[['name',
                   'accuracy',
                   'precision',
                   'recall',
                   'f1']])

RadImageNet loaded
Missing: ['fc.weight', 'fc.bias']
Unexpected: []

Training: RadImageNet Config A: Real Only
Class weights — healthy: 0.649, alzheimer: 2.175
  Epoch 5/20 | Loss: 0.4343
  Epoch 10/20 | Loss: 0.3403
  Epoch 15/20 | Loss: 0.2830
  Epoch 20/20 | Loss: 0.2090

--- RadImageNet Config A ---
Accuracy : 0.852  Precision: 0.630
Recall   : 0.850  F1       : 0.723
Confusion Matrix:
[[406  70]
 [ 21 119]]
RadImageNet loaded
Missing: ['fc.weight', 'fc.bias']
Unexpected: []

Training: RadImageNet Config B: Real + Synthetic
Class weights — healthy: 0.743, alzheimer: 1.531
  Epoch 5/20 | Loss: 0.4062
  Epoch 10/20 | Loss: 0.3051
  Epoch 15/20 | Loss: 0.2595
  Epoch 20/20 | Loss: 0.2034

--- RadImageNet Config B ---
Accuracy : 0.859  Precision: 0.650
Recall   : 0.821  F1       : 0.726
Confusion Matrix:
[[414  62]
 [ 25 115]]
RadImageNet loaded
Missing: ['fc.weight', 'fc.bias']
Unexpected: []

Training: RadImageNet Config C: Synthetic Only
Class weights — healthy: 0.593, alzheimer: 3.

In [ ]:
summary = pd.DataFrame(results)
summary = summary[['name', 'accuracy', 'precision', 'recall', 'f1']]
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(summary.to_string(index=False))

summary.to_csv(
    '/content/drive/MyDrive/ddpm-coronal-v7_output/classifier_results.csv',
    index=False
)
print("\n✅ Saved to Drive")

In [ ]:
# ====================================================================
# ──── FID SCORE CALCULATION USING RADIMAGENET FEATURE ───────────────
# ====================================================================

#calculate fid using radnet feature extractor
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from scipy import linalg
import os

# Feature extractor — RadImageNet ResNet50 without final FC layer
from torchvision import models
from huggingface_hub import hf_hub_download

class RadFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        # Create ResNet50 architecture
        model = models.resnet50(weights=None)

        # Download RadImageNet weights
        model_path = hf_hub_download(
            repo_id="Lab-Rasool/RadImageNet",
            filename="ResNet50.pt"
        )

        state_dict = torch.load(model_path, map_location="cpu")

        # Remap Sequential indices to torchvision's named layers
        index_to_name = {'0': 'conv1', '1': 'bn1', '4': 'layer1',
                          '5': 'layer2', '6': 'layer3', '7': 'layer4'}
        new_state_dict = {}
        for key, value in state_dict.items():
            if not key.startswith("backbone."):
                continue
            parts = key.replace("backbone.", "").split(".")
            if parts[0] in index_to_name:
                parts[0] = index_to_name[parts[0]]
                new_state_dict[".".join(parts)] = value

        # Load weights
        missing, unexpected = model.load_state_dict(
            new_state_dict,
            strict=False
        )

        print("RadImageNet loaded")
        print("Missing:", missing[:5])
        print("Unexpected:", unexpected[:5])

        # Convert RGB -> grayscale
        original_conv = model.conv1

        model.conv1 = nn.Conv2d(
            1,
            64,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )

        with torch.no_grad():
            model.conv1.weight.copy_(
                original_conv.weight.mean(dim=1, keepdim=True)
            )

        # Remove classifier
        self.features = nn.Sequential(
            *list(model.children())[:-1]
        )

    def forward(self, x):
        x = self.features(x)
        return x.flatten(1)

extractor = RadFeatureExtractor().to(device).eval()
print("Feature extractor ready ✅")

# Image loading
class ImgDataset(Dataset):
    def __init__(self, folder):
        self.files = [os.path.join(folder, f)
                      for f in os.listdir(folder) if f.endswith('.png')]
        self.tf = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.Grayscale(1),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        return self.tf(Image.open(self.files[idx]).convert('L'))

def extract_features(folder):
    loader = DataLoader(ImgDataset(folder), batch_size=32, shuffle=False)
    feats  = []
    with torch.no_grad():
        for batch in loader:
            feats.append(extractor(batch.to(device)).cpu().numpy())
    return np.concatenate(feats, axis=0)

def compute_fid(feats_real, feats_gen):
    """Compute FID between two feature arrays."""
    mu1, sig1 = feats_real.mean(0), np.cov(feats_real, rowvar=False)
    mu2, sig2 = feats_gen.mean(0),  np.cov(feats_gen,  rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sig1.dot(sig2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sig1 + sig2 - 2 * covmean)
    return float(fid)

# ── Compute FID with RadImageNet features ─────────────────
print("Extracting features — this takes a few minutes...\n")

real_100_dir = '/content/drive/MyDrive/Dissertation2026/ddpm-coronal-v7_output/real_split_a'
gen_run1_dir = '/content/drive/MyDrive/Dissertation2026/ddpm-coronal-v7_output/fid_generated_350_ddpm'

feats_real = extract_features(real_100_dir)
feats_gen1 = extract_features(gen_run1_dir)

# Real vs Real baseline
import random, shutil
real_b_dir = '/content/drive/MyDrive/ddpm-coronal-v7_output/real_split_b'   #ddpm_coronal_output
os.makedirs(real_b_dir, exist_ok=True)
if len(os.listdir(real_b_dir)) < 100:
    alz_dir = '/content/drive/MyDrive//Dissertation2026/oasis-coronal-alzheimers'
    files = [f for f in os.listdir(alz_dir) if f.endswith('.png')]
    random.seed(99)
    random.shuffle(files)
    used = set(os.listdir(real_100_dir))
    new  = [f for f in files if f not in used][:100]
    for f in new:
        shutil.copy(os.path.join(alz_dir, f), os.path.join(real_b_dir, f))

feats_real_b = extract_features(real_b_dir)

fid_baseline = compute_fid(feats_real, feats_real_b)
fid_run1     = compute_fid(feats_real, feats_gen1)

print("="*55)
print("FID WITH RADIMAGENET FEATURES")
print("="*55)
print(f"Real vs Real baseline : {fid_baseline:.2f}")
print(f"Generated vs Real     : {fid_run1:.2f}")
print(f"Gap                   : {fid_run1 - fid_baseline:.2f}")
print()
print("Previous FID (ImageNet features):")  #FID values from DDPM_coronal(linear) code
print(f"  Real vs Real: 5.40")
print(f"  Generated:    87.39")

RadImageNet loaded
Missing: ['fc.weight', 'fc.bias']
Unexpected: []
Feature extractor ready ✅
Extracting features — this takes a few minutes...



/tmp/ipykernel_546/3865175763.py:122: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(sig1.dot(sig2), disp=False)
/tmp/ipykernel_546/3865175763.py:122: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(sig1.dot(sig2), disp=False)


FID WITH RADIMAGENET FEATURES
Real vs Real baseline : 0.23
Generated vs Real     : 0.82
Gap                   : 0.59

Previous FID (ImageNet features):
  Real vs Real: 5.40
  Generated:    87.39
